<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">📘 Batch, micro-batch a streaming - Northwind </h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">V nadväznosti na Seminár 8 - Zadanie 1a s MovieLens spracujeme údaje o objednávkach. Tentoraz si jednotlivé kroky naprogramujete samostatne od načítania CSV po porovnanie výsledkov.</p>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Z kurzu si stiahnite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders.csv</code> (stĺpce <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderid</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">customerid</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">employeeid</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderdate</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">shipperid</code>). Súbor uložte do rovnakého priečinka ako notebook. Knižnice a PostgreSQL už máte pripravené.</p>

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 1 – Načítanie a časová distribúcia objednávok</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Načítajte súbor <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders.csv</code> a zmeňte <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderdate</code> na dátum. Zistite časový rozsah a počet objednávok podľa roku a mesiaca. Rozloženie zobrazte v grafe.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Pomôcka:</strong> <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">pd.read_csv(..., parse_dates=["orderdate"])</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">.dt.year</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">.dt.to_period("M")</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">value_counts()</code>.</div>

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# orders.csv je uložený v rovnakom priečinku ako notebook
CSV_PATH = Path("orders.csv")

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV sa nenašlo: {CSV_PATH.resolve()}. Uložte orders.csv vedľa notebooku.")

# parse_dates zmení orderdate z textu na datetime
orders = pd.read_csv(CSV_PATH, parse_dates=["orderdate"])

required = {"orderid", "customerid", "employeeid", "orderdate", "shipperid"}
missing = required - set(orders.columns)
if missing:
    raise ValueError(f"V CSV chýbajú stĺpce: {sorted(missing)}")

# časový rozsah datasetu
print("Počet objednávok:", len(orders))
print("Najstaršia objednávka:", orders["orderdate"].min())
print("Najnovšia objednávka:", orders["orderdate"].max())

# distribúcia podľa roku
orders_by_year = (
    orders["orderdate"]
    .dt.year
    .value_counts()
    .sort_index()
)
display(orders_by_year)

# distribúcia podľa mesiaca
orders_by_month = (
    orders["orderdate"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)
display(orders_by_month)

# vizualizácia – mesiacov je málo, preto stĺpcový graf
ax = orders_by_month.plot(
    kind="bar",
    figsize=(10, 3.5),
    color="#3577d4",
    title="Počet objednávok podľa mesiaca"
)
ax.set_xlabel("Mesiac")
ax.set_ylabel("Počet objednávok")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 2 – Historické a novšie objednávky</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Podľa časového rozloženia zvoľte <strong>hranice 1. januára 1997</strong>. Staršie objednávky budú historický batch, objednávky od tohto dátumu využijeme na simuláciu príchodu udalostí.</p>
<ul style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.7; margin-top: 6px; padding-left: 24px;"><li style="margin-bottom: 5px;"><code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">historical_orders</code> budú objednávky pred hranicou,</li><li style="margin-bottom: 5px;"><code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">new_orders</code> naopak objednávky od hranice (vrátane),</li><li style="margin-bottom: 5px;">overte, že obe skupiny spolu obsahujú všetky objednávky.</li></ul>
<div style="margin-top: 12px; background-color: #fefce8; border-left: 4px solid #f59e0b; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">⚠️ <strong>Upozornenie: </strong> „Novšie“ objednávky sú stále <strong>historické záznamy</strong>. V nasledujúcich úlohách budeme ich príchod len simulovať.</div>

In [ ]:
CUTOFF = pd.Timestamp("1997-01-01")

# historický batch: objednávky pred hranicou
historical_orders = orders[
    orders["orderdate"] < CUTOFF
].copy()

# novšie objednávky: od hranice (vrátane)
new_orders = orders[
    orders["orderdate"] >= CUTOFF
].copy()

print("Historické objednávky:", len(historical_orders))
print("Novšie objednávky:", len(new_orders))

# kontrola: každá objednávka patrí práve do jednej skupiny
assert len(historical_orders) + len(new_orders) == len(orders), \
    "Niektoré objednávky nepatria do žiadnej skupiny (napr. chýbajúci dátum)."
print("Spolu:", len(historical_orders) + len(new_orders), "= všetky objednávky ✔")

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 3 – Raw ingestion do PostgreSQL</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Vytvorte databázu <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">Northwind</code>, tabuľku <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders</code> s primárnym kľúčom <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderid</code> a nahrajte všetky záznamy z CSV. Predpokladáme, že PostgreSQL a balíky už fungujú inštaláciu ani test verzie preto nemusíte riešiť.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Pomôcka:</strong> najskôr sa pripojte do databázy <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">postgres</code>, vytvorte <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders_practice</code> s <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">AUTOCOMMIT</code>, potom zmeňte pripojenie. Na opakovateľný import použite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">DROP TABLE IF EXISTS</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">to_sql(..., if_exists="append", index=False)</code>.</div>
<div style="margin-top: 12px; background-color: #fefce8; border-left: 4px solid #f59e0b; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">⚠️ <strong> Upozornenie:</strong> Pri opakovanom spustení sa obsah tabuľky <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders</code> vymaže a znovu nahrá. Nepoužívajte názov databázy, ktorá už obsahuje vaše vlastné dáta.</div>

In [ ]:
from sqlalchemy import create_engine, text

# prihlasovacie údaje – heslo upravte podľa svojho nastavenia
DB_USER = "postgres"
DB_PASSWORD = "Student"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "northwind"   # PostgreSQL názvy bez úvodzoviek ukladá malými písmenami

BASE_URL = f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"

# 1) pripojíme sa k systémovej databáze postgres
engine = create_engine(f"{BASE_URL}/postgres")

# 2) vytvoríme databázu, ak ešte neexistuje
# AUTOCOMMIT je potrebný – CREATE DATABASE nemôže bežať v transakcii
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": DB_NAME}
    ).scalar()

    if not exists:
        conn.execute(text(f"CREATE DATABASE {DB_NAME}"))

# 3) prepneme pripojenie na novú databázu
engine.dispose()
engine = create_engine(f"{BASE_URL}/{DB_NAME}")

# 4) opakovateľný import – starú tabuľku zmažeme a vytvoríme znova
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS orders"))
    conn.execute(text("""
        CREATE TABLE orders (
            orderid    INT PRIMARY KEY,
            customerid INT,
            employeeid INT,
            orderdate  DATE,
            shipperid  INT
        )
    """))

# 5) nahráme všetky záznamy z CSV (raw ingestion – bez úprav)
orders.to_sql(
    "orders",
    engine,
    if_exists="append",
    index=False
)

# kontrola počtu riadkov v databáze
with engine.connect() as conn:
    db_count = conn.execute(text("SELECT COUNT(*) FROM orders")).scalar()

assert db_count == len(orders), "Počet riadkov v DB nesedí s CSV."
print("Tabuľka orders bola vytvorená a naplnená.")
print("Riadkov v databáze:", db_count)

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 4 – Batch processing</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Z historických objednávok vypočítajte <strong>počet objednávok</strong>, <strong>počet zákazníkov</strong> a <strong>percento objednávok s <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">shipperid = 3</code></strong>. Výsledok získame jednorazovo za celú historickú skupinu.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong> Poznámka:</strong> Percento vypočítate ako <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">(počet objednávok dopravcu 3 / počet všetkých objednávok) * 100</code>. Výraz <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">(df["shipperid"] == 3).sum()</code> spočíta vyhovujúce riadky.</div>

In [ ]:
# celý DataFrame historical_orders predstavuje jeden logický batch
hist_orders_count = len(historical_orders)
hist_customers = historical_orders["customerid"].nunique()
hist_shipper3 = (historical_orders["shipperid"] == 3).sum()
hist_shipper3_pct = hist_shipper3 / hist_orders_count * 100

print("Objednávok:", hist_orders_count)
print("Zákazníkov:", hist_customers)
print("Dopravca 3 (%):", round(hist_shipper3_pct, 2))

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 5 – Spoločná vzorka pre tri prístupy</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Na porovnanie musí každý prístup dostať <strong>rovnaké objednávky v rovnakom poradí</strong>. Z PostgreSQL vyberte novšie objednávky a zoraďte ich podľa <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderdate</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderid</code>.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong> Poznámka:</strong> Použite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">text()</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">engine.connect()</code> a pomenované parametre SQL <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">:cutoff</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">:limit</code>. V tomto datasete je po hranici 44 objednávok, preto postačuje <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">SAMPLE_SIZE = 44</code>.</div>

In [ ]:
# jedna spoločná vzorka pre batch, micro-batch aj streaming
SAMPLE_SIZE = 44

with engine.connect() as conn:
    order_sample = pd.read_sql(
        text("""
            SELECT orderid, customerid, employeeid, orderdate, shipperid
            FROM orders
            WHERE orderdate >= :cutoff
            ORDER BY orderdate, orderid
            LIMIT :limit
        """),
        conn,
        params={"cutoff": CUTOFF.date(), "limit": SAMPLE_SIZE},
        parse_dates=["orderdate"]
    )

if order_sample.empty:
    raise ValueError("Po hranici nie sú žiadne objednávky. Skontrolujte dáta alebo CUTOFF.")

print("Objednávok v simulácii:", len(order_sample))
print("Časový rozsah:", order_sample["orderdate"].min(), "–", order_sample["orderdate"].max())

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 6 – Simulácia micro-batchingu a streamingu</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Historické objednávky budeme prehrávať postupne. <strong>Streaming</strong> aktualizuje podiel dopravcu 3 po každej objednávke, <strong>micro-batch</strong> po uzavretí krátkeho časového okna. Batch poskytne výsledok až po príchode celej vzorky.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Poznámka:</strong> Pre tento malý dataset nastavíme <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">EVENT_DELAY_SEC = 0.04</code> (40 ms medzi objednávkami) a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">MICRO_WINDOW_SEC = 0.30</code> (približne 300 ms na dávku). Skutočné časy závisia od výkonu počítača.</div>

<h3 style="margin: 20px 0 10px; font-size: 17px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827;">Krok A – Nastavenia a priebežný stav</h3>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Pripravte časové parametre, počítadlá a zoznamy na zaznamenávanie priebežných výsledkov. Pri micro-batchingu potrebujeme navyše premenné pre neuzavretú dávku.</p>

In [ ]:
import time

# nastavenia experimentu
EVENT_DELAY_SEC = 0.04    # 40 ms medzi objednávkami
MICRO_WINDOW_SEC = 0.30   # približne 300 ms na jednu micro-batch dávku

# STREAMING – počet objednávok, z toho dopravca 3, história výsledkov
stream_count, stream_hits = 0, 0
stream_updates = []       # (čas, počet, percento)

# MICRO-BATCH – spracované a čakajúce (neuzavreté) objednávky
micro_count, micro_hits = 0, 0
micro_pending_count, micro_pending_hits = 0, 0
micro_updates = []        # (čas, počet, percento)

<h3 style="margin: 20px 0 10px; font-size: 17px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827;">Krok B – Spracovanie udalostí</h3>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Prejdite objednávky postupne. Pri streamingu priebežne zapisujte <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">(čas, počet, percento)</code>. Pri micro-batchingu spracujte nazbierané objednávky až po uplynutí okna.</p>

In [ ]:
# začiatok merania nastavujeme až tu, tesne pred príchodom prvej udalosti
start_time = time.perf_counter()
last_flush = start_time

# každá historická objednávka predstiera novú prichádzajúcu udalosť
for order in order_sample.itertuples(index=False):
    time.sleep(EVENT_DELAY_SEC)
    now = time.perf_counter()
    is_shipper3 = int(order.shipperid == 3)   # 1 = dopravca 3, inak 0

    # STREAMING – výsledok aktualizujeme po každej objednávke
    stream_count += 1
    stream_hits += is_shipper3
    stream_updates.append((now - start_time, stream_count,
                           stream_hits / stream_count * 100))

    # MICRO-BATCH – objednávku zatiaľ iba odložíme do otvoreného okna
    micro_pending_count += 1
    micro_pending_hits += is_shipper3

    # po uplynutí okna dávku spracujeme a zverejníme nový výsledok
    if now - last_flush >= MICRO_WINDOW_SEC:
        micro_count += micro_pending_count
        micro_hits += micro_pending_hits
        micro_updates.append((now - start_time, micro_count,
                              micro_hits / micro_count * 100))
        micro_pending_count, micro_pending_hits = 0, 0
        last_flush = now

# okamih príchodu poslednej objednávky
stream_end = time.perf_counter()

<h3 style="margin: 20px 0 10px; font-size: 17px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827;">Krok C – Posledná dávka a batch</h3>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Nezabudnite spracovať poslednú nedokončenú micro-batch dávku. Nakoniec vypočítajte jednorazový batch výsledok pre tú istú vzorku a overte rovnosť výsledných percent.</p>

In [ ]:
# posledná nedokončená micro-batch dávka
if micro_pending_count:
    micro_count += micro_pending_count
    micro_hits += micro_pending_hits
    micro_updates.append((stream_end - start_time, micro_count,
                          micro_hits / micro_count * 100))
    micro_pending_count, micro_pending_hits = 0, 0

# BATCH – jeden výsledok až po príchode celej vzorky
calc_start = time.perf_counter()
batch_count = len(order_sample)
batch_pct = (order_sample["shipperid"] == 3).sum() / batch_count * 100
batch_ready_sec = (stream_end - start_time) + (time.perf_counter() - calc_start)

stream_pct = stream_hits / stream_count * 100
micro_pct = micro_hits / micro_count * 100

print("Počet objednávok:", batch_count)
print("Batch (%):", round(batch_pct, 2))
print("Micro-batch (%):", round(micro_pct, 2))
print("Streaming (%):", round(stream_pct, 2))

# kontrola: všetky prístupy spracovali rovnaké objednávky a majú rovnaký výsledok
assert batch_count == micro_count == stream_count, "Počty objednávok sa líšia."
assert abs(batch_pct - micro_pct) < 1e-9 and abs(batch_pct - stream_pct) < 1e-9, \
    "Výsledné percentá sa líšia."
print("Všetky tri prístupy dali rovnaký výsledok ✔")
print("Micro-batch aktualizácií:", len(micro_updates),
      "| Streaming aktualizácií:", len(stream_updates))

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 7 – Grafy priebežných výsledkov</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Vytvorte <strong>tri samostatné grafy</strong> pod sebou. V prvom zobrazte jednorazový batch bod, v druhom schodovitý priebeh micro-batchingu a v treťom streamingovú krivku. Použite spoločné osi, aby boli percentá porovnateľné.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Poznámka:</strong> Použite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">plt.subplots(3, 1, sharex=True, sharey=True)</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">scatter()</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">step(where="post")</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">plot()</code>.</div>

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, sharey=True)

# 1. BATCH – jeden bod na konci
axes[0].scatter([batch_ready_sec], [batch_pct], color="#3577d4", s=100, zorder=5)
axes[0].set_title("Batch – jeden výsledok po príchode celej vzorky")

# 2. MICRO-BATCH – schodovitý priebeh po dávkach
axes[1].step(
    [x[0] for x in micro_updates],
    [x[2] for x in micro_updates],
    where="post", color="#159572", linewidth=2
)
axes[1].scatter([x[0] for x in micro_updates], [x[2] for x in micro_updates],
                color="#159572", s=20)
axes[1].set_title("Micro-batch – aktualizácia po uzavretí okna")

# 3. STREAMING – krivka po každej objednávke
axes[2].plot(
    [x[0] for x in stream_updates],
    [x[2] for x in stream_updates],
    color="#c88730", linewidth=1.5
)
axes[2].set_title("Event streaming – aktualizácia po každej objednávke")

# spoločné nastavenia – referenčná čiara s konečným výsledkom
for ax in axes:
    ax.axhline(batch_pct, color="gray", linestyle="--", linewidth=1)
    ax.set_ylabel("Dopravca 3 (%)")
    ax.grid(alpha=0.2)

axes[2].set_xlabel("Čas od začiatku simulácie (s)")
plt.tight_layout()
plt.show()

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 8 – Frekvencia aktualizácií a prvý výsledok</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Na troch grafoch znázornite, koľkokrát sa priebežný výsledok aktualizoval. Potom vytvorte jednoduchú tabuľku, ktorá porovná <strong>čas prvého výsledku</strong>.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong> Poznámka:</strong> Pri počte aktualizácií nie je vhodné zdieľať os Y: batch má jeden výsledok, streaming jeden po každej udalosti. Os X zdieľajte.</div>

In [ ]:
# os Y nezdieľame (1 vs. desiatky aktualizácií), os X áno
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# 1. BATCH – výsledok iba raz
axes[0].step([0, batch_ready_sec], [0, 1], where="post", color="#3577d4", linewidth=2)
axes[0].scatter([batch_ready_sec], [1], color="#3577d4", s=70, zorder=5)
axes[0].set_title("Batch")

# 2. MICRO-BATCH – po každej dávke
axes[1].step(
    [0] + [x[0] for x in micro_updates],
    [0] + list(range(1, len(micro_updates) + 1)),
    where="post", color="#159572", linewidth=2
)
axes[1].set_title("Micro-batch")

# 3. STREAMING – po každej objednávke
axes[2].step(
    [0] + [x[0] for x in stream_updates],
    [0] + list(range(1, len(stream_updates) + 1)),
    where="post", color="#c88730", linewidth=1.8
)
axes[2].set_title("Event streaming")

for ax in axes:
    ax.set_ylabel("Počet aktualizácií")
    ax.grid(alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

axes[2].set_xlabel("Čas od začiatku simulácie (s)")
plt.tight_layout(h_pad=2)
plt.show()

# tabuľka: čas prvého výsledku a počet aktualizácií
results = pd.DataFrame({
    "Prístup": ["Batch", "Micro-batch", "Event streaming"],
    "Prvý výsledok (s)": [batch_ready_sec, micro_updates[0][0], stream_updates[0][0]],
    "Počet aktualizácií": [1, len(micro_updates), len(stream_updates)],
})

print(results.to_string(index=False, formatters={
    "Prvý výsledok (s)": "{:.3f}".format
}))

fastest = results.loc[results["Prvý výsledok (s)"].idxmin(), "Prístup"]
print("\nPrvý výsledok najskôr poskytol:", fastest)

<div style="margin-top: 12px; background-color: #f0fdf4; border-left: 4px solid #22c55e; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">
✅ <strong>Vyhodnotenie:</strong> Najskôr dá výsledok <strong>streaming</strong> – hneď po prvej objednávke (≈ 40 ms). <strong>Micro-batch</strong> čaká na uzavretie prvého okna (≈ 300 ms) a <strong>batch</strong> až na príchod všetkých 44 objednávok (≈ 44 × 40 ms ≈ 1,8 s). Konečné percento dopravcu 3 je vo všetkých prípadoch rovnaké – líši sa iba to, <strong>kedy</strong> a <strong>ako často</strong> je výsledok dostupný. Na začiatku streamu sa percento výrazne mení, pretože je vypočítané iba z pár objednávok.
</div>

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 9 – Micro-batch po pevnom počte objednávok (<code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">range()</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">iloc</code>)</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Micro-batch môžeme tvoriť aj podľa <strong>počtu záznamov</strong> namiesto času. Vzorku rozdelíme na dávky po <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">BATCH_SIZE</code> objednávok: <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">range(0, len(order_sample), BATCH_SIZE)</code> vráti začiatky dávok (0, 10, 20, …) a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">iloc[start:start + BATCH_SIZE]</code> vyberie riadky dávky podľa pozície.</p>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Po každej dávke aktualizujeme <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">running_count</code> (počet objednávok) a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">running_sum</code> (počet objednávok dopravcu 3). Podiel dopravcu 3 je vlastne <strong>priemer</strong> stĺpca, v ktorom má dopravca 3 hodnotu 1 a ostatní 0 – preto <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">running_avg = running_sum / running_count</code>, v percentách × 100.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Pomôcka:</strong> 44 objednávok nie je deliteľných 10 – posledná dávka bude mať iba 4 záznamy a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">iloc</code> vráti jednoducho zvyšok.</div>

In [ ]:
BATCH_SIZE = 10

# priebežný stav – stačia dve čísla
running_count = 0     # počet spracovaných objednávok
running_sum = 0       # z toho objednávky dopravcu 3
micro_batch_results = []

# range() vráti začiatok každej dávky: 0, 10, 20, ...
for start in range(0, len(order_sample), BATCH_SIZE):
    # iloc vyberie riadky dávky podľa pozície
    batch = order_sample.iloc[start:start + BATCH_SIZE]
    batch_hits = int((batch["shipperid"] == 3).sum())

    running_count += len(batch)
    running_sum += batch_hits
    running_avg = running_sum / running_count          # priebežný priemer (0–1)

    micro_batch_results.append({
        "dávka": start // BATCH_SIZE + 1,
        "riadky": f"{start}–{start + len(batch) - 1}",
        "od": batch["orderdate"].min().date(),
        "do": batch["orderdate"].max().date(),
        "v dávke": len(batch),
        "dopravca 3 v dávke": batch_hits,
        "running_count": running_count,
        "running_sum": running_sum,
        "running_avg (%)": round(running_avg * 100, 2),
    })

micro_batch_df = pd.DataFrame(micro_batch_results)
display(micro_batch_df)

print("Konečný running_avg (%):", round(running_avg * 100, 2))
print("Batch (%):              ", round(batch_pct, 2))

# kontrola – nič sa nestratilo ani nezdvojilo a výsledok sa zhoduje s batchom
assert running_count == len(order_sample)
assert abs(running_avg * 100 - batch_pct) < 1e-9
print("Micro-batch (range + iloc) = batch ✔")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))

# podiel dopravcu 3 v jednotlivých dávkach vs. priebežný podiel
batch_share = micro_batch_df["dopravca 3 v dávke"] / micro_batch_df["v dávke"] * 100
ax.bar(micro_batch_df["dávka"], batch_share, color="#cbd5e1", label="Podiel v dávke")
ax.plot(micro_batch_df["dávka"], micro_batch_df["running_avg (%)"],
        marker="o", color="#159572", linewidth=2, label="running_avg")
ax.axhline(batch_pct, color="#3577d4", linestyle="--", linewidth=1, label="Batch")

ax.set_xticks(micro_batch_df["dávka"])
ax.set_xlabel("Číslo dávky")
ax.set_ylabel("Dopravca 3 (%)")
ax.set_title(f"Micro-batch po {BATCH_SIZE} objednávkach")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

<div style="margin-top: 12px; background-color: #f0fdf4; border-left: 4px solid #22c55e; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">
✅ <strong>Záver:</strong> Podiel dopravcu 3 sa medzi dávkami výrazne líši, no priebežný <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">running_avg</code> sa s pribúdajúcimi dávkami ustáli a po poslednej dávke sa presne zhoduje s batch výsledkom. Na priebežný výpočet nepotrebujeme uchovávať všetky predchádzajúce objednávky – stačí <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">running_count</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">running_sum</code>.
</div>

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">
  📘 Kontrola Git repozitára po 8. týždni
</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">
  Po dokončení všetkých úloh z 8. týždňa skontrolujte, či máte vo svojom Git
  repozitári vytvorenú samostatnú vetvu
  <span style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">week-08</span>
  a či sa v nej nachádza vypracovaný notebook vytvorený počas cvičenia.
</p>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">
  Obsah vetvy
  <span style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">week-08</span>
  by mal mať približne nasledujúcu štruktúru:
</p>
<pre style="background: #f9fafb; color: #111827; padding: 14px; border-radius: 7px; overflow: auto; font-size: 13px; line-height: 1.7; font-family: ui-monospace, SFMono-Regular, Menlo, monospace;">week-08
│
├── ratings.csv
├── orders.csv
│
├── batch_streaming.ipynb
└── batch_streaming_orders.ipynb
</pre>
<div style="margin-top: 12px; background-color: #fefce8; border-left: 4px solid #eab308; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">
  ⚠️ <strong>Kontrola GitHubu:</strong> Skontrolujte, či sa všetky uvedené súbory
  nachádzajú aj vo vzdialenom repozitári na GitHube. Ak ich vidíte iba lokálne
  vo VS Code, zmeny ešte neboli úspešne odoslané pomocou
  <strong>commit</strong> a <strong>push</strong>.
</div>